# Django Forms

En este ejercicio se creó un ModelForm para registrar nuevos usuarios
en el eCommerce.

El formulario permite capturar username, email, contraseña y confirmación
de contraseña. La contraseña se almacena correctamente utilizando
set_password().


## ModelForm de registro

In [ ]:
from django import forms
from django.contrib.auth import get_user_model


User = get_user_model()


class UserRegistrationForm(forms.ModelForm):
    password = forms.CharField(
        label="Contraseña",
        widget=forms.PasswordInput,
    )

    password_confirmation = forms.CharField(
        label="Confirmar contraseña",
        widget=forms.PasswordInput,
    )

    class Meta:
        model = User
        fields = [
            "username",
            "email",
            "password",
        ]

    def clean(self):
        cleaned_data = super().clean()

        password = cleaned_data.get("password")
        password_confirmation = cleaned_data.get(
            "password_confirmation"
        )

        if password != password_confirmation:
            self.add_error(
                "password_confirmation",
                "Las contraseñas no coinciden.",
            )

        return cleaned_data

    def save(self, commit=True):
        user = super().save(commit=False)
        user.set_password(
            self.cleaned_data["password"]
        )

        if commit:
            user.save()

        return user


## Vista de registro

In [ ]:
from django.shortcuts import redirect, render

from .forms import UserRegistrationForm


def register(request):
    if request.method == "POST":
        form = UserRegistrationForm(request.POST)

        if form.is_valid():
            form.save()
            return redirect("login")

    else:
        form = UserRegistrationForm()

    return render(
        request,
        "ejercicios/register.html",
        {"form": form},
    )


## URLs

In [ ]:
from django.contrib.auth.views import LoginView
from django.urls import path
from ejercicios.views import register


urlpatterns = [
    path(
        "register/",
        register,
        name="register"
    ),

    path(
        "accounts/login/",
        LoginView.as_view(
            template_name="ejercicios/login.html"
        ),
        name="login"
    ),
]


## Template de registro

In [ ]:
<h1>Crear una cuenta</h1>

<form method="post">
    {% csrf_token %}

    {{ form.as_p }}

    <button type="submit">
        Registrarse
    </button>
</form>


## Pruebas realizadas

El ModelForm se probó múltiples veces desde el navegador.

Primera prueba:
- Usuario: prueba1
- El registro se procesó correctamente.
- El usuario apareció en Django Admin.

Segunda prueba:
- Usuario: prueba2
- El registro se procesó correctamente.
- El usuario apareció en Django Admin.

Después de las pruebas, Django Admin mostró 4 usuarios en total:
Mark, Ryu, prueba1 y prueba2.

Esto comprobó que UserRegistrationForm crea correctamente nuevos
usuarios en la base de datos.
